# einops-reduce — ex7: per-channel BN-style stats (mean, var, normalized output)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-reduce`. Running the final beacon cell reports progress against the `Einops: Reduce` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Reduce` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-reduce`** (exercise 7). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-reduce"
DD_SUBTOPIC = "Einops: Reduce"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## einops.reduce — quick refresher

`reduce(tensor, pattern, op)` collapses one or more named axes with a reduction `op` ∈ `{'mean', 'sum', 'max', 'min', 'prod'}`. Drop an axis name on the right side to reduce it; keep it inside parentheses on the left and decompose first to do windowed pooling.

The exercises below stop being about *which op?* and start being about *reduce as part of a larger pipeline* — pyramid pooling, per-channel normalization, argmax-without-`torch.argmax`, top-k by repeated masked max. Each one needs visualization or print-debug to be solvable in your head.

### Exercise 7 — per-channel BN-style stats (mean, var, normalized output)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Use einops.reduce with `keepdim`-style `()` axes to compute per-channel mean+var over (batch, height, width), then broadcast back to normalize, and visualize the per-channel result.
> Keywords: batchnorm, broadcast, keepdim, normalization
> ```

**KCs targeted:** `reduce-mean`, `reduce-keepdim-with-parens`

BatchNorm in 2-D normalizes each channel independently using statistics gathered across the `(batch, height, width)` axes. Done naively with `.mean(dim=...)`, the result has the wrong shape for broadcasting back. Done with `reduce`'s `()` keepdim trick, the broadcast just works.

Implement `ex7_per_channel_bn(x, eps=1e-5)`:
1. `x` has shape `(N, C, H, W)`.
2. Compute `mu` of shape `(1, C, 1, 1)` using one `reduce` call with `()` on the b/h/w slots. Same for `var` (use the formula `E[x²] - E[x]²` so you only do `reduce`-style ops).
3. Return `(x - mu) / sqrt(var + eps)` — shape `(N, C, H, W)`.
4. **Print** `mu` and `var` flattened (one row per channel), then plot the per-channel mean of the normalized output as a bar chart (should all be ≈ 0) and the per-channel variance as another bar chart (should all be ≈ 1).

These two bar charts are the BatchNorm sanity check — if they're not flat at 0 and 1, your reduce broadcast is wrong.

In [ ]:
def ex7_per_channel_bn(x: Tensor, eps: float = 1e-5) -> Tensor:
    import matplotlib.pyplot as plt
    # The () axes keep that slot in the output with size 1 — perfect for broadcast.
    mu = reduce(x, 'n c h w -> () c () ()', 'mean')
    mean_sq = reduce(x ** 2, 'n c h w -> () c () ()', 'mean')
    var = mean_sq - mu ** 2
    y = (x - mu) / t.sqrt(var + eps)
    print('per-channel mu :', mu.flatten().tolist())
    print('per-channel var:', var.flatten().tolist())
    # Sanity bar charts.
    out_mean = reduce(y, 'n c h w -> c', 'mean')
    out_var = reduce(y ** 2, 'n c h w -> c', 'mean') - out_mean ** 2
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3))
    ax1.bar(range(len(out_mean)), out_mean.cpu().numpy())
    ax1.set_title('output mean per channel (should be ~0)')
    ax1.axhline(0, color='k', linewidth=0.5)
    ax2.bar(range(len(out_var)), out_var.cpu().numpy())
    ax2.set_title('output var per channel (should be ~1)')
    ax2.axhline(1, color='k', linewidth=0.5)
    plt.tight_layout()
    plt.show()
    return y


<details><summary>Solution</summary>

```python
def ex7_per_channel_bn(x: Tensor, eps: float = 1e-5) -> Tensor:
    import matplotlib.pyplot as plt
    # The () axes keep that slot in the output with size 1 — perfect for broadcast.
    mu = reduce(x, 'n c h w -> () c () ()', 'mean')
    mean_sq = reduce(x ** 2, 'n c h w -> () c () ()', 'mean')
    var = mean_sq - mu ** 2
    y = (x - mu) / t.sqrt(var + eps)
    print('per-channel mu :', mu.flatten().tolist())
    print('per-channel var:', var.flatten().tolist())
    # Sanity bar charts.
    out_mean = reduce(y, 'n c h w -> c', 'mean')
    out_var = reduce(y ** 2, 'n c h w -> c', 'mean') - out_mean ** 2
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3))
    ax1.bar(range(len(out_mean)), out_mean.cpu().numpy())
    ax1.set_title('output mean per channel (should be ~0)')
    ax1.axhline(0, color='k', linewidth=0.5)
    ax2.bar(range(len(out_var)), out_var.cpu().numpy())
    ax2.set_title('output var per channel (should be ~1)')
    ax2.axhline(1, color='k', linewidth=0.5)
    plt.tight_layout()
    plt.show()
    return y
```

**Why `()` in the pattern?** Writing `'n c h w -> () c () ()'` tells einops: collapse n, h, w but *keep* a length-1 slot in their positions. The result is shape `(1, C, 1, 1)`, which broadcasts against `(N, C, H, W)` with zero ceremony. Without the `()`, you'd get shape `(C,)` and need a manual `view(1, C, 1, 1)` — that's the silent shape bug `()` exists to prevent.

**Why E[x²] − E[x]²?** It lets you compute variance using only reduce-style ops, no `.var` call. Numerically less stable than the two-pass formula for huge tensors, but fine here and demonstrates that variance *is* just two reductions plus a subtract.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex7'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex7',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()